# 04 回测
Backtrader 回测，LightGBM vs Ridge 对比

In [ ]:
import sys, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.insert(0, os.path.abspath('..'))
from src.models import predict_scores
from src.clustering import cluster_stocks
from src.backtest import run_backtest
from src.utils import calc_sharpe, calc_max_drawdown, calc_annual_return

%matplotlib inline
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)

In [ ]:
df = pd.read_parquet('../data/processed_data.parquet')
df = cluster_stocks(df, n_clusters=4)
df['cluster_feat'] = df['cluster'].fillna(-1).astype(float)

feature_cols = ['factor_pe', 'factor_momentum_20', 'factor_vol_60', 'cluster_feat']
target_col = 'fwd_ret_5d'

print(f'数据加载完成: {df.shape}')

In [ ]:
# 生成预测得分并回测
backtest_results = {}

for model_type in ['lgb', 'ridge']:
    print(f'\n=== {model_type.upper()} 回测 ===')
    df_scores = predict_scores(df, feature_cols=feature_cols, target_col=target_col, model_type=model_type, train_window=504)
    cerebro, summary = run_backtest(df_scores, top_pct=0.2, commission=0.001)
    backtest_results[model_type] = summary
    
    print(f'  总收益: {summary["total_return"]*100:.2f}%')
    if summary['annual_return']:
        print(f'  年化收益: {summary["annual_return"]:.2f}%')
    if summary['sharpe']:
        print(f'  夏普: {summary["sharpe"]:.4f}')
    print(f'  最大回撤: {summary["max_drawdown"]:.2f}%')

In [ ]:
# 净值曲线对比
plt.figure(figsize=(14, 6))

colors = {'lgb': '#2E86AB', 'ridge': '#A23B72'}

for name, summary in backtest_results.items():
    nav = summary['nav']
    if not nav.empty:
        plt.plot(nav.index, nav.values, label=f"{name.upper()}", color=colors.get(name, 'gray'), linewidth=1.5)

plt.xlabel('Date')
plt.ylabel('Portfolio Value')
plt.title('净值曲线对比 (Initial Cash = 1,000,000)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# 回测指标汇总
metrics = []
for name, summary in backtest_results.items():
    metrics.append({
        'Model': name.upper(),
        'Total Return': f"{summary['total_return']*100:.2f}%",
        'Annual Return': f"{summary['annual_return']:.2f}%" if summary['annual_return'] else 'N/A',
        'Sharpe': f"{summary['sharpe']:.4f}" if summary['sharpe'] else 'N/A',
        'Max DD': f"{summary['max_drawdown']:.2f}%",
        'Final Value': f"{summary['final_value']:,.0f}",
    })

metrics_df = pd.DataFrame(metrics)
metrics_df